# Agents as tools

Read the [guide](README.md). Install the notebooks, multiagent and partner groups for the full tier. All model calls below replay offline; local A2A transports still use real child processes.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_specialist_tools

Predict the result, then run the [script](examples/01_specialist_tools.py).


In [ ]:
"""Specialists wrapped as tools: Nimbus Tier 4 lesson.

Run: uv run python modules/15_agents_as_tools/examples/01_specialist_tools.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio
import json

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.specialists import concierge


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: the orchestrator can call only the three host-bound specialist tools.
    result = asyncio.run(concierge(cfg))
    print(json.dumps(result, indent=2))
    complete()


run_lesson(main, "modules/15_agents_as_tools/examples/01_specialist_tools.py")

## 02_context_boundaries

Predict the result, then run the [script](examples/02_context_boundaries.py).


In [ ]:
"""Minimum context and authoritative output validation: Nimbus Tier 4 lesson.

Run: uv run python modules/15_agents_as_tools/examples/02_context_boundaries.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.specialists import specialist
from common.trip_planning import TripRequest, catalog, validate_offer


def main() -> None:
    _, cfg = lesson_settings()
    request = TripRequest()
    records = []
    answer = asyncio.run(specialist(cfg, "flight", request, records))
    print("Context fields:", sorted(records[0]["context"]))
    print("Validated answer:", answer)
    # KEY IDEA: a schema alone cannot establish that a price is true.
    try:
        validate_offer({**answer, "cents": 1}, catalog("flight", request))
    except ValueError as exc:
        print("Rejected altered quote:", str(exc))
    complete()


run_lesson(main, "modules/15_agents_as_tools/examples/02_context_boundaries.py")

## 03_delegation_cost

Predict the result, then run the [script](examples/03_delegation_cost.py).


In [ ]:
"""Compare delegation with a single agent: Nimbus Tier 4 lesson.

Run: uv run python modules/15_agents_as_tools/examples/03_delegation_cost.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio
import json

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.specialists import concierge


def main() -> None:
    _, cfg = lesson_settings()
    rows = [asyncio.run(concierge(cfg, delegated=value)) for value in (False, True)]
    for row in rows:
        row.pop("children")
    # KEY IDEA: count parent AND child usage; a scripted answer is no quality benchmark.
    print(json.dumps(rows, indent=2))
    print("Score real answers for reference accuracy, policy coverage and usefulness.")
    complete()


run_lesson(main, "modules/15_agents_as_tools/examples/03_delegation_cost.py")

## nimbus_concierge

Predict the result, then run the [script](use_case/nimbus_concierge.py).


In [ ]:
"""Nimbus concierge: Nimbus Tier 4 lesson.

Run: uv run python modules/15_agents_as_tools/use_case/nimbus_concierge.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio
import json

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.specialists import concierge


def main() -> None:
    _, cfg = lesson_settings()
    result = asyncio.run(concierge(cfg))
    print(json.dumps(result, indent=2))
    print("Public quotes only; no bookings, customer records or payment tools.")
    complete()


run_lesson(main, "modules/15_agents_as_tools/use_case/nimbus_concierge.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md). Module 17 defaults to rejection; use the standalone --decision ask command to review and approve a draft.
